# Problem: Implement Logistic Regression

### Problem Statement
Your task is to implement a **Logistic Regression** model using PyTorch. The model should predict a binary class label based on a given set of input features.

### Requirements
1. **Model Definition**:
   - Implement a class `LogisticRegressionModel` with:
     - A single linear layer mapping input features to one output.
     - A sigmoid activation to produce a probability.
2. **Forward Method**:
   - Implement the `forward` method to compute predictions given input data.

### Part 1: Using `nn.Module`
### Part 2: Using Raw Tensor Operations

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Generate synthetic binary classification data
torch.manual_seed(42)
X = torch.randn(200, 2)  # 200 data points, 2 features
y = ((X[:, 0] + X[:, 1]) > 0).float().unsqueeze(1)  # label 1 if sum > 0

## Part 1: Logistic Regression with `nn.Module`

In [5]:
# Define the Logistic Regression Model
# TODO: Add the layer and forward implementation
class LogisticRegressionModel(nn.Module):
    def __init__(self):
        super(LogisticRegressionModel, self).__init__()
        # TODO: Define a linear layer (2 inputs, 1 output)
        self.linear = nn.Linear(2,1)

    def forward(self, x):
        # TODO: Apply linear layer then sigmoid
        return torch.sigmoid(self.linear(x))

# Initialize the model, loss function, and optimizer
model = LogisticRegressionModel()
# TODO: Use BCELoss for binary cross-entropy
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 0.2996
Epoch [200/1000], Loss: 0.2252
Epoch [300/1000], Loss: 0.1923
Epoch [400/1000], Loss: 0.1724
Epoch [500/1000], Loss: 0.1586
Epoch [600/1000], Loss: 0.1484
Epoch [700/1000], Loss: 0.1403
Epoch [800/1000], Loss: 0.1337
Epoch [900/1000], Loss: 0.1282
Epoch [1000/1000], Loss: 0.1235


In [7]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weights: {w.data}, Learned bias: {b.item():.4f}")

# Compute accuracy
with torch.no_grad():
    preds = (model(X) >= 0.5).float()
    accuracy = (preds == y).float().mean()
    print(f"Training accuracy: {accuracy.item():.4f}")

# Test on new data
X_test = torch.tensor([[1.0, 1.0], [-1.0, -1.0]])
with torch.no_grad():
    probs = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {probs.tolist()}")

Learned weights: tensor([[3.7608, 3.6881]]), Learned bias: -0.1570
Training accuracy: 0.9950
Predictions for [[1.0, 1.0], [-1.0, -1.0]]: [[0.9993194341659546], [0.0004972329479642212]]


## Part 2: Logistic Regression with Raw Tensor Operations

Implement the same model **without** `nn.Module`, `nn.Linear`, or `optim`. Use only tensors, sigmoid, and autograd.

In [23]:
# TODO: Initialize weight w (2x1) and bias b with requires_grad=True
w = torch.randn(2, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Training loop
lr = 0.1
epochs = 1000
for epoch in range(epochs):
    # TODO: Forward pass with sigmoid: torch.sigmoid(X @ w + b)
    y_pred = torch.sigmoid(torch.matmul(X,w) + b)

    # TODO: Binary cross-entropy loss manually
    # -(y * log(y_pred) + (1-y) * log(1-y_pred)).mean()
    # Hint: add 1e-7 inside log() for numerical stability
    # loss = -(y * torch.log(y_pred + 1e-7) + (1 - y) * torch.log(1 - y_pred + 1e-7)).mean()

    # loss_func = nn.BCELoss()
    # loss = loss_func(y_pred,y)

    loss = -1*(y * torch.log(y_pred + 1e-7) + (1 - y) * torch.log(1-y_pred + 1e-7)).mean()

    # Backward pass
    loss.backward()

    # TODO: Update parameters manually inside torch.no_grad()
    with torch.no_grad():
        w -= w.grad * lr
        b -= b.grad * lr

    # TODO: Zero gradients manually
    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/1000], Loss: 0.3190
Epoch [200/1000], Loss: 0.2301
Epoch [300/1000], Loss: 0.1947
Epoch [400/1000], Loss: 0.1740
Epoch [500/1000], Loss: 0.1598
Epoch [600/1000], Loss: 0.1492
Epoch [700/1000], Loss: 0.1410
Epoch [800/1000], Loss: 0.1343
Epoch [900/1000], Loss: 0.1287
Epoch [1000/1000], Loss: 0.1239


In [21]:
# Display the learned parameters
print(f"Learned weights: {w.data.squeeze().tolist()}, Learned bias: {b.item():.4f}")

# Compute accuracy
with torch.no_grad():
    preds = (torch.sigmoid(X @ w + b) >= 0.5).float()
    accuracy = (preds == y).float().mean()
    print(f"Training accuracy: {accuracy.item():.4f}")

# Test on new data
X_test = torch.tensor([[1.0, 1.0], [-1.0, -1.0]])
with torch.no_grad():
    probs = torch.sigmoid(X_test @ w + b)
    print(f"Predictions for {X_test.tolist()}: {probs.tolist()}")

Learned weights: [3.806270122528076, 3.734666585922241], Learned bias: -0.1601
Training accuracy: 0.9950
Predictions for [[1.0, 1.0], [-1.0, -1.0]]: [[0.9993773102760315], [0.000452165404567495]]
